**In this Notebook we will do the following:**

- Construct the Pauli-Transfer-Matrix (PTM) of multiple Circuits
    - Memory Round: Surface Code
    - Lattice Surgery

- Invert the PTM to get an estimate of the Logical Operator

In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys, pathlib
repo_root = pathlib.Path.cwd()

# If running from the playground directory, move up one level to the repo root
if repo_root.name == 'playground':
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))
print('Inserted repo root into sys.path:', repo_root)

Inserted repo root into sys.path: C:\Users\f.spreemann\qecsim-work


In [7]:
from src.core.data_models import NoiseParameters
from src.tools.qem_estimator.logical_level.calc_ptm import PTMCalculator
from src.core.data_models import PTMCircuits
from src.codes.lattice_surgery.builder import SurgeryBuilder
from itertools import product
from tqdm.notebook import tqdm

import stim

**PTM Calculation: Lattice Surgery**

In [8]:
#########################
# Construct Noise Class #
#########################
noise = 0

noise_class = NoiseParameters(before_round_depol = noise,
    before_m_flip_prob = noise,
    after_r_flip = noise,
    after_c_depol_prob = noise,
    after_c_pauli_channel_prob = noise)

############
# Circuits #
############

# Pauli alphabet for input and output states
PAULIS = ["I", "X", "Y", "Z"]

# Create Mapping for input states
input_to_init_state = {
    "X": ["X+", "X-"],
    "Y": ["Y+", "Y-"],
    "Z": ["Z0", "Z1"],
    "I": ["Z0", "Z1"],
}

# Create Mapping for measurement basis
pauli_to_measure_basis = {
    "X": "X",
    "Y": "Y",
    "Z": "Z",
    "I": "Z"
}

circuits_surgery : dict[str, tuple[dict[str,stim.Circuit], list[int]]] = {}

# We combine the products so the progress bar tracks all 256 combinations
total_combinations = list(product(PAULIS, PAULIS, PAULIS, PAULIS))

for p_out_c, p_out_t, p_in_c, p_in_t in tqdm(total_combinations, desc="Generating Circuits"):

    # Initialize current measurement records and Circuit list
    curr_meas_rec: list[int] = []
    all_circuits: list[stim.Circuit] = []

    # Creating Label
    label_basis = f"{p_in_c}{p_in_t}->{p_out_c}{p_out_t}"

    # Creating inner dict
    inner_dict: dict[str, stim.Circuit] = {}
    
    # Inner loop for the 4 initial states
    for init_state_c, init_state_t in product(input_to_init_state[p_in_c], input_to_init_state[p_in_t]):

        # Creating state Label
        state_label = f"{init_state_c},{init_state_t}"

        builder = SurgeryBuilder(
            distance = 3, 
            control_state_init=init_state_c, 
            target_state_init=init_state_t, 
            control_measure_basis=pauli_to_measure_basis[p_out_c],
            target_measure_basis=pauli_to_measure_basis[p_out_t], 
            noise=noise_class
        )
        
        circuit = builder.build_circuit()
        all_circuits.append(circuit)

        # It doesnt matter which records we get as the logical observable stays the same
        # in this loop
        curr_meas_rec = builder.get_logical_meas_rec(observable_index=0)

        # Adding the circuit to the inner dict
        inner_dict[state_label] = circuit

    # Adding to your existing dict
    circuits_surgery[label_basis] = (inner_dict, curr_meas_rec)

Generating Circuits:   0%|          | 0/256 [00:00<?, ?it/s]

In [9]:
import numpy as np
np.set_printoptions(precision=4, suppress=True, linewidth=200)

############################
# Calculate the PTM-Matrix #
############################

ptm_calculator = PTMCalculator(PTMCircuits(circuits=circuits_surgery), samples=1_000)
ptm_matrix_surgery = ptm_calculator.calc_ptm(only_non_zero=True)

#get_sgn = ptm_calculator._get_sgn("XI->XX")
#get_sgn
ptm_matrix_surgery

array([[ 0.07 ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   , -0.068],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.032,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   , -0.08 ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ,  0.   ],
       [ 0.   ,  0.   ,  0.   ,  0.   ,  